# 2.1 Eşik ayarlama: HLS-L / LAB-B

`renk_esikleme.py` ile **aynı** eşikleme fonksiyonunu (`color_masks`) kullanır; burada bulduğun değerler scriptte birebir aynı sonucu verir.

1. **AYARLAR** hücresinde kendi kareni seç (görüntü ya da video + kare numarası).
2. Kanal görüntüleri ve histogramlar, çizgilerin hangi değerlerde ayrıldığını gösterir.
3. Kaydırıcılarla eşikleri dene. Hedef: çizgiler tam görünsün, asfalt, beton, gökyüzü ve gölge maskeye girmesin.
4. En alttaki çıktıdaki satırları `renk_esikleme.py` dosyasının AYARLAR bölümüne yaz ya da komut satırı argümanı olarak ver.

Notebook'u kendi klasöründen aç (Jupyter: http://127.0.0.1:8890); script aynı klasörden import edilir.

In [ ]:
# AYARLAR
try:
    import renk_esikleme as esik  # aynı klasördeki script (başka klasörden import yok)
except ModuleNotFoundError:
    raise SystemExit("HATA: renk_esikleme.py bulunamadı.\n"
                     "  -> Notebook'u kendi klasöründen (2.1 klasörü) aç.") from None

# [DEĞİŞTİR] Denenecek kare: görüntü ya da video (videoda FRAME_INDEX. kare).
IMAGE_PATH = esik.INPUT_PATH
FRAME_INDEX = 0

In [ ]:
import cv2
import matplotlib.pyplot as plt

try:
    frame = esik.load_frame(IMAGE_PATH, FRAME_INDEX)
except esik.InputError as exc:
    raise SystemExit(f"HATA: {exc}") from None

hls = cv2.cvtColor(frame, cv2.COLOR_BGR2HLS)
lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
channels = {"HLS-L (açıklık)": hls[:, :, 1], "HLS-S (doygunluk)": hls[:, :, 2],
            "LAB-B (mavi <-> sarı)": lab[:, :, 2]}

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes[0, 0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title(f"Girdi {frame.shape[1]}x{frame.shape[0]}")
axes[1, 0].axis("off")
for i, (name, channel) in enumerate(channels.items(), start=1):
    axes[0, i].imshow(channel, cmap="gray", vmin=0, vmax=255)
    axes[0, i].set_title(name)
    axes[1, i].hist(channel.ravel(), bins=256, range=(0, 256), log=True)
    axes[1, i].set_title(f"{name} histogramı (log)")
for ax in axes[0]:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets


def show(white_channel, white_l_range, white_s_range, yellow_b_range):
    white_range = white_l_range if white_channel == "L" else white_s_range
    white, yellow, mask = esik.color_masks(frame, white_channel, white_range, yellow_b_range)

    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
    images = (cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), white, yellow, mask)
    titles = ("Girdi", f"Beyaz: HLS-{white_channel} {white_range[0]}-{white_range[1]}",
              f"Sarı: LAB-B {yellow_b_range[0]}-{yellow_b_range[1]}", "Birleşik maske (OR)")
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    plt.show()

    print("Maske oranı:", esik.format_ratios(white, yellow, mask))
    print("\nAYARLAR'a yazılacak satırlar (renk_esikleme.py):")
    print(f'WHITE_CHANNEL = "{white_channel}"')
    print(f"WHITE_L_RANGE = {tuple(white_l_range)}")
    print(f"WHITE_S_RANGE = {tuple(white_s_range)}")
    print(f"YELLOW_B_RANGE = {tuple(yellow_b_range)}")
    print("\nKomut satırı:")
    print(f"python renk_esikleme.py --white-channel {white_channel} "
          f"--white-l-range {white_l_range[0]} {white_l_range[1]} "
          f"--white-s-range {white_s_range[0]} {white_s_range[1]} "
          f"--yellow-b-range {yellow_b_range[0]} {yellow_b_range[1]}")


# [EK] ipywidgets kaydırıcıları: eşikleri canlı denemek için (jupyter paketiyle kurulu).
slider = dict(min=0, max=255, step=1, continuous_update=False,
              layout=widgets.Layout(width="500px"))
widgets.interact(
    show,
    white_channel=widgets.ToggleButtons(options=["L", "S"], value=esik.WHITE_CHANNEL,
                                        description="Beyaz kanal"),
    white_l_range=widgets.IntRangeSlider(value=esik.WHITE_L_RANGE, description="HLS-L", **slider),
    white_s_range=widgets.IntRangeSlider(value=esik.WHITE_S_RANGE, description="HLS-S", **slider),
    yellow_b_range=widgets.IntRangeSlider(value=esik.YELLOW_B_RANGE, description="LAB-B", **slider),
);

### İpuçları
- **Beyaz çizgi (HLS-L):** Alt eşik çok düşükse açık renkli beton ve gökyüzü (L ≈ 185-190) maskeye girer, çok yüksekse eskimiş boya (L ≈ 200) kaybolur. Gölgedeki beyaz boyanın L değeri (≈ 140) açık asfaltla aynı olduğu için yalnızca renk eşiğiyle ayrılamaz; gölgeye karşı yöntemler 4.x gereksinimlerinde.
- **Sarı çizgi (LAB-B):** 128 nötrdür. Soluk ya da gölgedeki sarı ≈ 160-185, kum ve toprak ≈ 150-155 aralığındadır. Çim (≈ 164) ve kırmızı/turuncu nesneler de yüksek B değerine sahiptir; bunlar yol dışında kaldığı için perspektif/ROI adımında elenir.
- **HLS-S:** Beyaz boyanın S değeri ≈ 0-12'dir; yüksek S eşiği beyazı değil sarıyı ve gökyüzünü yakalar. Karşılaştırmak için *Beyaz kanal* düğmesini S yap.